# Averaged across problems (≥ cutoff): Rank Proportions around good reversals

This notebook reproduces your per-problem analysis:

- `get_good_reversal_info(..., include_first_block=True)`
- `get_rank_counts_by_good_reversal(reversal_windows)`
- `pvalue_paired_t_best_vs_second_vs_third(rank_counts_by_good_reversal)`
- `plot_rank_proportions(...)`

…but instead of generating one figure per problem, it **averages across problems ≥ `CUTOFF_PROBLEM`**.

### How averaging works here
`rank_counts_by_good_reversal` is a dict keyed by mouse/subject, containing per-reversal aligned rank-proportion/count data.
To average across problems, we **concatenate the per-problem reversal windows per subject** (i.e., treat each reversal instance as another sample),
then pass the pooled structure into `plot_rank_proportions` and the p-value function.

This is safe because rank counts are computed per reversal window and don't rely on global trial indices across problems.

In [7]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.behavior_import.import_data import *
from src.behavior_import.extract_trials import *
from src.behavior_analysis.get_good_reversal_info import *
from src.behavior_analysis.get_rank_counts_by_good_reversal import *
from src.behavior_analysis.get_diagnostic_p_value import *
from src.behavior_visualization.plot_rank_proportions import *
from fix_grid_maze_cohort_02_problems import *

import numpy as np


## Parameters

In [8]:
task = "grid-maze"
# task = "open-field"

CUTOFF_PROBLEM = 7  # average problems >= this

OUT_BASE = Path("../results/figures")


## Load data and group into problems

In [9]:
folder_name = None
cohort = None
if task == "grid-maze":
    cohort = "cohort-02"
    folder_name = "3x3_maze_blocked_reward_bandit"
elif task == "open-field":
    cohort = "cohort-01"
    folder_name = "3x3_field_blocked_reward_bandit"

root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"

subjects_data = import_data(root)
subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)

if task == "grid-maze" and cohort == "cohort-02":
    subjects_trials_by_problem = fix_grid_maze_cohort_02_problems(subjects_trials_by_problem)

problem_numbers = sorted(subjects_trials_by_problem.keys())
probs_used = [p for p in problem_numbers if p >= CUTOFF_PROBLEM]
print("Problems found:", problem_numbers)
print("Averaging problems >=", CUTOFF_PROBLEM, ":", probs_used)


[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-27_date-20260124/behav/._MY_04_R-2026-01-24-124422.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-55_date-20260209/behav/._MY_04_R-2026-02-09-103918.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-28_date-20260124/behav/._MY_04_R-2026-01-24-165301.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-12_date-20260116/behav/._MY_04_R-2026-01-16-143640.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volum

## Build and merge reversal windows across problems ≥ cutoff

In [10]:
def merge_windows_by_subject(windows_by_problem):
    """windows_by_problem[p][subj] = list[reversal_dict]. Returns merged[subj] = concatenated list."""
    merged = {}
    for p, win in windows_by_problem.items():
        if win is None:
            continue
        for subj, revs in win.items():
            if not revs:
                continue
            merged.setdefault(subj, []).extend(revs)
    return merged

windows_by_problem = {}
for p in sorted(subjects_trials_by_problem.keys()):
    if p < CUTOFF_PROBLEM:
        continue
    subjects_trials = subjects_trials_by_problem[p]
    rw = get_good_reversal_info(subjects_trials, include_first_block=True)
    windows_by_problem[p] = rw

reversal_windows_merged = merge_windows_by_subject(windows_by_problem)

print("Merged subjects:", len(reversal_windows_merged))
print("Total merged reversals:", sum(len(v) for v in reversal_windows_merged.values()))


Merged subjects: 6
Total merged reversals: 265


## Rank counts + p-values + plot (averaged across problems)

In [11]:
import numpy as np
from pathlib import Path
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Helvetica Neue", "Helvetica", "Arial"]
mpl.rcParams["xtick.labelsize"] = 12
mpl.rcParams["ytick.labelsize"] = 12
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 14

MOUSE_COLORS = [
    "#4C72B0","#55A868","#C44E52","#8172B2",
    "#CCB974","#64B5CD","#8C8C8C","#DD8452",
    "#937860","#DA8BC3","#8C6D31","#1F77B4",
]

BLOCK_COLORS = [
    "#1B9E77", "#D95F02", "#7570B3", "#E7298A",
    "#66A61E", "#E6AB02", "#A6761D", "#666666",
    "#1F78B4", "#33A02C",
    "#FB9A99", "#E31A1C", "#FDBF6F", "#CAB2D6",
    "#6A3D9A", "#FFFF99", "#B15928", "#A6CEE3",
    "#B2DF8A", "#FF7F00",
]

def plot_rank_proportions_average_without_blocks(rank_counts_by_good_reversal, average_across_mice_pvalues=None, save_path=None):
    """
    Two-panel summary:
      (Left)  Average across mice: bars = mean across mice, lines = individual mice (mouse colors).
      (Right) Average across blocks: bars = mean across blocks, lines = individual blocks (block colors).

    rank_counts_by_good_reversal[subj] should be a list of dicts with keys:
      best_prop, second_prop, third_prop, total, ...
    """

    subjects = list(rank_counts_by_good_reversal.keys())
    labels = ["Best", "Second", "Third"]
    x = np.arange(len(labels))
    jitter = 0.03

    fig, axes = plt.subplots(1, 1, figsize=(13, 5.5), sharey=True)

    # ============================================================
    # LEFT: Average across mice
    # ============================================================
    ax = axes

    per_mouse = {}
    for subj in subjects:
        rows = rank_counts_by_good_reversal.get(subj, [])
        if not rows:
            continue

        per_mouse[subj] = {
            "best": np.nanmean([r.get("best_prop", np.nan) for r in rows]),
            "second": np.nanmean([r.get("second_prop", np.nan) for r in rows]),
            "third": np.nanmean([r.get("third_prop", np.nan) for r in rows]),
        }

    mouse_means = [
        np.nanmean([v["best"] for v in per_mouse.values()]),
        np.nanmean([v["second"] for v in per_mouse.values()]),
        np.nanmean([v["third"] for v in per_mouse.values()]),
    ]
    mouse_se = [
        np.nanstd([v[k] for v in per_mouse.values()], ddof=1) / np.sqrt(len(per_mouse))
        for k in ["best", "second", "third"]
    ]

    ax.bar(x, mouse_means, yerr=mouse_se, capsize=6, width=0.55, color="#999999", edgecolor="black", linewidth=1.5, alpha=0.55, zorder=0)

    if average_across_mice_pvalues:
        p_bs = average_across_mice_pvalues.get("best_vs_second", None)
        p_bt = average_across_mice_pvalues.get("best_vs_third", None)
        p_st = average_across_mice_pvalues.get("second_vs_third", None)
    else:
        p_bs = p_bt = p_st = None

    xi_s = []
    for xi, m in zip(x, mouse_means):
        if np.isfinite(m):
            ax.text(xi, 1.05, f"{m:.2f}", ha="center", va="bottom", fontsize=12)
            xi_s.append(xi)
    
    if p_bs is not None:
        ax.text(xi_s[0], 0.75, f"p(Best vs Second):\n{p_bs:.3f}", ha="center", va="bottom", fontsize=12)
    if p_bt is not None:
        ax.text(xi_s[2], 0.25, f"p(Best vs Third):\n{p_bt:.3f}", ha="center", va="bottom", fontsize=12)
    if p_st is not None:
        ax.text(xi_s[1], 0.5, f"p(Second vs Third):\n{p_st:.3f}", ha="center", va="bottom", fontsize=12)
    
    legend_handles = []
    for i, (subj, v) in enumerate(per_mouse.items()):
        c = MOUSE_COLORS[i % len(MOUSE_COLORS)]
        y = [v["best"], v["second"], v["third"]]
        ax.plot(x + np.random.uniform(-jitter, jitter, size=len(x)), y, color=c, linewidth=2.5, marker="o", alpha=0.9, markersize=6,)
        legend_handles.append(Line2D([0], [0], color=c, lw=2.5, marker="o", label=subj))

    ax.set_title("Average Across Mice", pad=14)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Proportion of Choices")
    ax.set_ylim(0.0, 1.07)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(handles=legend_handles, fontsize=10, loc="upper right")

    if save_path:
        fig.savefig(save_path + ".pdf", bbox_inches="tight")
        fig.savefig(save_path + ".png", dpi=300, bbox_inches="tight")
    else:
        plt.show()
    plt.close(fig)

In [12]:
rank_counts_by_good_reversal = get_rank_counts_by_good_reversal(reversal_windows_merged)
p_values = pvalue_paired_t_best_vs_second_vs_third(rank_counts_by_good_reversal)

save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "choice-stats" / "Rank Proportions (Averaged)"
plot_rank_proportions_average_without_blocks(rank_counts_by_good_reversal, average_across_mice_pvalues=p_values, save_path=str(save_path))
